# Titanic: Machine Learning from Disaster

**Goal:** predict whether each passenger survived the sinking of the Titanic (binary classification).
**Metric:** accuracy on 418 test passengers.

**Final result:** 0.791 on the Kaggle leaderboard (gradient boosting with engineered features).

| Submission | What changed | Score |
|---|---|---|
| Baseline | "Women survive" rule | 0.766 |
| Random forest | Title, age fill, family size, cabin flag | 0.778 |
| Gradient boosting | + group survival feature | **0.791** |

**Main lesson:** almost all of the gain came from features that turn human knowledge into numbers (titles, family groups), not from swapping models.

## 1. Setup and data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
print(train.shape, test.shape)
train.head()

`train.csv` has 891 passengers with the `Survived` label (the textbook with answers). `test.csv` has 418 passengers without it (the exam).

Missing values to deal with:

In [ ]:
pd.DataFrame({"train": train.isnull().sum(), "test": test.isnull().sum()}).query("train > 0 or test > 0")

## 2. Exploratory data analysis

Before building anything, look at who survived. Each `groupby` asks: *of everyone in this group, what fraction survived?*

In [ ]:
print(f"Overall survival rate: {train['Survived'].mean():.1%}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["Sex", "Pclass", "Embarked"]):
    sns.barplot(data=train, x=col, y="Survived", ax=ax, errorbar=None)
    ax.set_title(f"Survival rate by {col}")
    ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

**Sex** is the strongest signal: about 74% of women survived versus 19% of men ("women and children first").
**Class** comes next: 63% in 1st, 47% in 2nd, 24% in 3rd — first-class cabins were closer to the lifeboats.

In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(data=train, x="Age", hue="Survived", bins=30, multiple="stack")
plt.title("Age distribution by survival")
plt.show()

Young children (under ~10) survived at a much higher rate, while young adult men had the worst odds.

## 3. Baseline: "women survive"

Before modelling, measure the simplest rule. Every model afterwards has to beat this number — like timing your 5K before starting a training plan.

In [ ]:
baseline = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": (test["Sex"] == "female").astype(int),
})
print("Baseline accuracy on train:", ((train["Sex"] == "female").astype(int) == train["Survived"]).mean().round(4))
# Kaggle score: 0.76555

## 4. Cleaning and feature engineering

Train and test are combined so both go through exactly the same steps — two batches of dough through the same recipe. `ignore_index=True` keeps row numbers unique (0–1308).

In [ ]:
full = pd.concat([train, test], ignore_index=True, sort=False)

### 4.1 Title from Name

Names follow the pattern **Surname, Title. First names**. The regex grabs whatever sits between the comma and the period. French titles are merged into English ones, and rare titles (Dr, Rev, Col, Countess…) are pooled, because a model can't learn from one or two examples.

In [ ]:
full["Title"] = full["Name"].str.extract(r",\s*([^\.]+)\.")[0]
full["Title"] = full["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
full.loc[~full["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Title"] = "Rare"
full["Title"].value_counts()

In [ ]:
sns.barplot(data=full[full["Survived"].notna()], x="Title", y="Survived", errorbar=None,
            order=["Mr", "Master", "Rare", "Miss", "Mrs"])
plt.title("Survival rate by title")
plt.ylim(0, 1)
plt.show()

"Master" (young boys) survived far more often than "Mr", even though both are male. Title captures the "children first" effect that the Sex column alone misses.

### 4.2 Missing values

- **Age:** filled with the median age *for that title*. A "Master" is a young boy, so he gets the typical Master's age (~4), not the overall average (~28) — like guessing someone's age from "Grandpa" versus "kiddo".
- **Embarked / Fare:** only 2 and 1 missing, so simple fills (most common port, median fare).
- **Cabin:** 77% empty, so instead of inventing values we record *whether* a cabin was listed (correlates with wealth).

In [ ]:
full["Age"] = full["Age"].fillna(full.groupby("Title")["Age"].transform("median"))
full["Embarked"] = full["Embarked"].fillna(full["Embarked"].mode()[0])
full["Fare"] = full["Fare"].fillna(full["Fare"].median())
full["HasCabin"] = full["Cabin"].notna().astype(int)

### 4.3 Family features

`FamilySize` counts the passenger plus siblings/spouses and parents/children. Travelling alone and very large families both did worse than small families of 2–4: a couple can move fast together, a family of 8 is hard to gather on a sinking ship.

In [ ]:
full["FamilySize"] = full["SibSp"] + full["Parch"] + 1
full["IsAlone"] = (full["FamilySize"] == 1).astype(int)

sns.barplot(data=full[full["Survived"].notna()], x="FamilySize", y="Survived", errorbar=None)
plt.title("Survival rate by family size")
plt.show()

### 4.4 Group survival

Families and ticket-sharing groups mostly lived or died together. For each passenger, this feature is the average survival of the *other* members of their group whose outcome is known.

- Groups are identified by **Surname + Fare** (a household address — surname alone mixes unrelated Smiths), then by **Ticket** to catch friends and servants.
- **0.5** means "no information" (alone, or group members all in test).
- Each passenger's own label is removed (`drop(idx)`) — otherwise the model would be handed the answer, like a student grading their own exam.

In [ ]:
full["Surname"] = full["Name"].str.split(",").str[0]
full["FamilyGroup"] = full["Surname"] + "_" + full["Fare"].astype(str)
full["GroupSurvival"] = 0.5

for col in ["FamilyGroup", "Ticket"]:
    for _, grp in full.groupby(col):
        if len(grp) > 1:
            for idx in grp.index:
                if full.at[idx, "GroupSurvival"] != 0.5:
                    continue
                others = grp["Survived"].drop(idx).dropna()
                if len(others) > 0:
                    full.at[idx, "GroupSurvival"] = others.mean()

print("Test passengers with a group signal:", (full.loc[891:, "GroupSurvival"] != 0.5).sum(), "of 418")

## 5. Encoding

Models need numbers. `Sex` becomes 0/1. Columns with more than two options are one-hot encoded with `get_dummies` — checkboxes instead of a free-text field — so the model doesn't assume an order (e.g. Miss "bigger" than Mr).

In [ ]:
full["Sex"] = full["Sex"].map({"male": 0, "female": 1})

features = ["Pclass", "Sex", "Age", "Fare", "Embarked", "Title",
            "FamilySize", "IsAlone", "HasCabin", "GroupSurvival"]
X_all = pd.get_dummies(full[features], columns=["Embarked", "Title"])

X, X_test = X_all[:891], X_all[891:]
y = train["Survived"]
X.head()

## 6. Model comparison

5-fold cross-validation trains on 4/5 of the data and tests on the remaining 1/5, five times — like taking five mock exams before the real one. `StandardScaler` puts features on the same scale for logistic regression and SVM (tree models don't need it).

In [ ]:
models = {
    "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42),
    "GradBoost": GradientBoostingClassifier(random_state=42),
    "SVM": make_pipeline(StandardScaler(), SVC()),
}

results = {name: cross_val_score(m, X, y, cv=5).mean() for name, m in models.items()}
pd.Series(results).sort_values(ascending=False).round(4)

Without `GroupSurvival`, all four models scored 0.825–0.836: different algorithms, same ceiling, meaning the features were the limit. Adding group survival lifted gradient boosting to ~0.85 in CV.

**Caveat:** this CV score is somewhat optimistic. During CV, some training passengers' group feature was built from the labels of the passengers being tested. The Kaggle leaderboard (0.791) is the honest measure.

## 7. What the model learned

In [ ]:
final_model = GradientBoostingClassifier(random_state=42)
final_model.fit(X, y)

importances = pd.Series(final_model.feature_importances_, index=X.columns).sort_values()
importances.plot.barh(figsize=(8, 6), title="Feature importance (gradient boosting)")
plt.show()

## 8. Submission

In [ ]:
predictions = final_model.predict(X_test).astype(int)
submission = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": predictions})
submission.to_csv("submission.csv", index=False)
print(submission.shape)
submission.head()

## 9. Conclusions

- **Features beat models.** Four very different algorithms reached nearly identical scores until a new feature added new information.
- **Domain knowledge pays.** The biggest gains came from encoding historical facts: "women and children first" (Sex, Title), wealth and location on the ship (Pclass, Fare, HasCabin), and families staying together (GroupSurvival).
- **CV vs leaderboard.** Cross-validation on 891 rows was consistently optimistic (~0.85 vs 0.791). Trusting the leaderboard for every small tweak risks overfitting to the 418 test passengers.

**Possible next steps:** hyperparameter tuning for gradient boosting, removing weak features, and averaging several models.